In [1]:
from pathlib import Path as P

import pandas as pd

OUTPUT_DIR = P("results/my_notebooks/computational_requirements")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
results_dir = P("results/my_notebooks")

# GLM fitting time + n_nuclei
df_fit_time = pd.read_csv(results_dir / "computational_requirements/time.csv")
# GLM destriping time
df_destripe_time = pd.read_csv(results_dir / "computational_requirements/time_destriping.csv")
# GLM fitting memory
df_fit_mem = pd.read_csv(results_dir / "peak_memory/peak_memory_fitting.csv")
# GLM destriping memory
df_destripe_mem = pd.read_csv(results_dir / "peak_memory/peak_memory_destriping.csv")
# B2C time and memory
df_b2c_time = pd.read_csv(results_dir / "b2c_baseline_requirements/time.csv")
df_b2c_mem = pd.read_csv(results_dir / "b2c_baseline_requirements/peak_memory.csv")

In [25]:
# Build summary table keyed on dataset
df = (
    df_fit_time[["dataset", "n_nuclei"]]
    .merge(df_fit_mem[["run", "peak_memory_GB"]].rename(columns={"run": "dataset", "peak_memory_GB": "fitting_mem"}), on="dataset")
    .merge(df_fit_time[["dataset", "fitting_time (min)"]].rename(columns={"fitting_time (min)": "fitting_time"}), on="dataset")
    .merge(df_destripe_mem[["run", "peak_memory_GB"]].rename(columns={"run": "dataset", "peak_memory_GB": "destriping_mem"}), on="dataset")
    .merge(df_destripe_time[["dataset", "destriping_time (min)"]].rename(columns={"destriping_time (min)": "destriping_time"}), on="dataset")
    .merge(df_b2c_mem[["run", "peak_memory_GB"]].rename(columns={"run": "dataset", "peak_memory_GB": "b2c_mem"}), on="dataset")
    .merge(df_b2c_time[["dataset", "time_destripe_min"]].rename(columns={"time_destripe_min": "b2c_time"}), on="dataset")
)

# Sort by number of nuclei and format dataset names
df = df.sort_values("n_nuclei").reset_index(drop=True)
df["dataset"] = df["dataset"].str.replace("_", " ")

# Build MultiIndex columns
df = df.set_index("dataset")
memory_heading = r"memory (GB)"
time_heading = r"time (min)"
summary = pd.DataFrame(
    {
        ("", "# nuclei"): df["n_nuclei"],
        ("ours (fitting)", memory_heading): df["fitting_mem"],
        ("ours (fitting)", time_heading): df["fitting_time"],
        ("ours (destriping)", memory_heading): df["destriping_mem"],
        ("ours (destriping)", time_heading): df["destriping_time"],
        ("B2C", memory_heading): df["b2c_mem"],
        ("B2C", time_heading): df["b2c_time"],
    }
)
summary.columns = pd.MultiIndex.from_tuples(summary.columns)
summary.index.name = ""
summary

ours (fitting)            ours (destriping)  \
                 # nuclei    memory (GB) time (min)       memory (GB)   
                                                                        
mouse brain         61842           0.26      39.45              5.17   
zebrafish head     121779           0.15     108.15              3.10   
human lymph node   292813           1.05     253.50              3.79   
mouse embryo       448092           1.46     222.55              8.13   

                                    B2C             
                 time (min) memory (GB) time (min)  
                                                    
mouse brain            0.53        2.61       0.07  
zebrafish head         0.15        1.19       0.02  
human lymph node       0.25        0.89       0.03  
mouse embryo           0.77        3.66       0.11

In [26]:
# Save CSV
summary.to_csv(OUTPUT_DIR / "summary_table.csv")

# Save LaTeX
# latex = summary.to_latex(
#     float_format="%.2f",
#     multicolumn_format="c",
#     column_format="|l|c|cc|cc|cc|",
#     escape=False,
# )
# latex = latex.replace("# nuclei", r"\# nuclei")
# latex = (
#     latex.replace(r"\toprule", r"\hline")
#     .replace(r"\midrule", r"\hline")
#     .replace(r"\bottomrule", r"\hline")
# )

# latex = (
#     latex.replace(
#         r"\multicolumn{2}{c}{ours (fitting)}", r"\multicolumn{2}{c|}{ours (fitting)}"
#     )
#     .replace(
#         r"\multicolumn{2}{c}{ours (destriping)}",
#         r"\multicolumn{2}{|c|}{ours (destriping)}",
#     )
#     .replace(r"\multicolumn{2}{c}{B2C}", r"\multicolumn{2}{|c|}{B2C}")
# )

summary = summary.rename(
    columns={
        "# nuclei": r"\# nuclei",
        "memory (GB)": r"\shortstack{memory\\(GB)}",
        "time (min)": r"\shortstack{time\\(min)}",
    }
)
latex = (
    summary.style.format(precision=2)
    .set_table_styles(
        [
            {"selector": "toprule", "props": ":hline;"},
            {"selector": "midrule", "props": ":hline;"},
            {"selector": "bottomrule", "props": ":hline;"},
        ],
        overwrite=False,
    )
    .to_latex(
        column_format="|l|r|rr|rr|rr|",
        multicol_align="|c|",
        hrules=True,
    )
)
(OUTPUT_DIR / "summary_table.tex").write_text(latex)
print(latex)

\begin{tabular}{|l|r|rr|rr|rr|}
\toprule
 &  & \multicolumn{2}{|c|}{ours (fitting)} & \multicolumn{2}{|c|}{ours (destriping)} & \multicolumn{2}{|c|}{B2C} \\
 & \# nuclei & \shortstack{memory\\(GB)} & \shortstack{time\\(min)} & \shortstack{memory\\(GB)} & \shortstack{time\\(min)} & \shortstack{memory\\(GB)} & \shortstack{time\\(min)} \\
 &  &  &  &  &  &  &  \\
\midrule
mouse brain & 61842 & 0.26 & 39.45 & 5.17 & 0.53 & 2.61 & 0.07 \\
zebrafish head & 121779 & 0.15 & 108.15 & 3.10 & 0.15 & 1.19 & 0.02 \\
human lymph node & 292813 & 1.05 & 253.50 & 3.79 & 0.25 & 0.89 & 0.03 \\
mouse embryo & 448092 & 1.46 & 222.55 & 8.13 & 0.77 & 3.66 & 0.11 \\
\bottomrule
\end{tabular}

